# Aceleración en GPU (CuPy y Numba CUDA) — Colab

Este notebook demuestra la parte de GPU NVIDIA del proyecto de tráfico aéreo.
En los Macs (M1 / M1 Pro) el proyecto usa Numba en CPU o MLX; aquí se mide CuPy y Numba CUDA.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).

In [ ]:
!nvidia-smi

## 1. Subir el código del proyecto
Comprime la carpeta `src` del proyecto (`zip -r src.zip src`) y súbela con la celda siguiente.

In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()  # selecciona src.zip
for name in up:
    zipfile.ZipFile(name).extractall(".")
print(os.listdir("."))

## 2. Verificar que CuPy ve la GPU

In [ ]:
import numpy as np, cupy as cp
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

from src.compute.backend import available_backends
print("Backends disponibles:", available_backends())

## 3. Benchmark de los backends (rutas simuladas)

In [ ]:
import pandas as pd
from src.compute.backend import benchmark

times = benchmark(n=1_000_000, n_steps=64)
df = pd.DataFrame({"backend": list(times), "segundos": list(times.values())})
df["aceleración vs numpy"] = df.loc[df.backend == "numpy", "segundos"].iloc[0] / df["segundos"]
df

In [ ]:
df.plot.bar(x="backend", y="segundos", legend=False, title="Tiempo para 1,000,000 de rutas")

## 4. Comprobar que la GPU da el mismo resultado que numpy

In [ ]:
from src.compute.backend import get_backend

rng = np.random.default_rng(0)
n = 100_000
lat1, lat2 = rng.uniform(15, 32, n), rng.uniform(15, 32, n)
lon1, lon2 = rng.uniform(-117, -86, n), rng.uniform(-117, -86, n)
speed = rng.uniform(600, 900, n)

ref = get_backend("numpy").simulate_routes(lat1, lon1, lat2, lon2, speed, 32)
gpu = get_backend("cupy").simulate_routes(lat1, lon1, lat2, lon2, speed, 32)
print("Coinciden:", np.allclose(ref, gpu))

## 5. Kernel propio con Numba CUDA
La misma distancia (haversine) escrita como kernel CUDA explícito: un hilo por vuelo.

In [ ]:
import math
from numba import cuda

@cuda.jit
def haversine_kernel(lat1, lon1, lat2, lon2, out):
    i = cuda.grid(1)
    if i < out.size:
        d2r = math.pi / 180.0
        dlat = (lat2[i] - lat1[i]) * d2r
        dlon = (lon2[i] - lon1[i]) * d2r
        a = math.sin(dlat / 2) ** 2 + math.cos(lat1[i] * d2r) * math.cos(lat2[i] * d2r) * math.sin(dlon / 2) ** 2
        out[i] = 2 * 6371.0088 * math.asin(math.sqrt(min(a, 1.0)))

d_in = [cuda.to_device(a) for a in (lat1, lon1, lat2, lon2)]
d_out = cuda.device_array(n)
threads = 256
blocks = (n + threads - 1) // threads
haversine_kernel[blocks, threads](*d_in, d_out)
cuda_km = d_out.copy_to_host()

ref_km = get_backend("numpy").haversine_km(lat1, lon1, lat2, lon2)
print("Numba CUDA coincide con numpy:", np.allclose(cuda_km, ref_km))

## 6. Conclusión
Anota aquí los tiempos de la tabla del punto 3 y compáralos con los de tu Mac
(`Cómputo` → «Correr benchmark» en el dashboard de Streamlit).